# 🔥 Forest Fire Risk RS
## Система аналізу ризику лісових пожеж

Цей ноутбук дозволяє завантажувати супутникові знімки для аналізу лісових пожеж в Україні.

**Можливості:**
- Вибір області з відкритого джерела GeoBoundaries
- Три режими завантаження: вся область, власна зона інтересу, лише території з пожежами
- Автоматичне завантаження порівняльних знімків (попередні місяці та попередній рік)
- Інтеграція з NASA FIRMS та Planet Labs
- Детальне логування (без збереження API ключів)

In [ ]:
# Встановлення залежностей
!pip install -q geopandas requests ipywidgets shapely fiona pyproj rtree ipyfilechooser folium

In [ ]:
import os
import json
import zipfile
import requests
import geopandas as gpd
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from dataclasses import dataclass, field
from io import BytesIO
import uuid
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from ipyfilechooser import FileChooser
import folium
from shapely.geometry import shape, box
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Підключення Google Drive (опційно)
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)

## Конфігурація та допоміжні класи

In [ ]:
@dataclass
class ComparisonConfig:
    """Налаштування порівняльних знімків"""
    include_previous_months: bool = True
    previous_months_count: int = 2
    include_previous_year: bool = True
    previous_years_count: int = 1
    
    def generate_periods(self, start_date: datetime, end_date: datetime) -> List[Dict]:
        """Генерує періоди для порівняння"""
        periods = []
        
        # Попередні місяці
        if self.include_previous_months:
            for i in range(1, self.previous_months_count + 1):
                prev_start = start_date - relativedelta(months=i)
                prev_end = end_date - relativedelta(months=i)
                periods.append({
                    'type': 'previous_month',
                    'offset': i,
                    'start': prev_start,
                    'end': prev_end,
                    'description': f'{i} міс. тому',
                    'folder_suffix': f'prev_{i}_months'
                })
        
        # Ті ж місяці в попередньому році
        if self.include_previous_year:
            for i in range(1, self.previous_years_count + 1):
                prev_start = start_date - relativedelta(years=i)
                prev_end = end_date - relativedelta(years=i)
                periods.append({
                    'type': 'previous_year',
                    'offset': i,
                    'start': prev_start,
                    'end': prev_end,
                    'description': f'Ті ж місяці {i} р. тому',
                    'folder_suffix': f'prev_{i}_year'
                })
        
        return periods


@dataclass
class SessionLog:
    """Лог сесії завантаження (без API ключів!)"""
    session_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    started_at: datetime = field(default_factory=datetime.now)
    region_name: str = ""
    download_mode: str = ""
    output_folder: str = ""
    fire_period: Optional[Tuple[datetime, datetime]] = None
    comparison_config: Optional[ComparisonConfig] = None
    aoi_file_name: Optional[str] = None
    downloaded_files: List[Dict] = field(default_factory=list)
    firms_data_file: Optional[str] = None
    boundary_file: Optional[str] = None
    errors: List[str] = field(default_factory=list)
    data_sources: List[str] = field(default_factory=list)
    
    def add_file(self, filename: str, file_type: str, size_bytes: int = 0):
        self.downloaded_files.append({
            'filename': filename,
            'type': file_type,
            'size_bytes': size_bytes,
            'downloaded_at': datetime.now().isoformat()
        })
    
    def add_error(self, error: str):
        self.errors.append(f"[{datetime.now().strftime('%H:%M:%S')}] {error}")
    
    def to_log_string(self) -> str:
        """Генерує текстовий лог (без API ключів!)"""
        lines = [
            "=" * 70,
            "           FOREST FIRE RISK RS - DOWNLOAD LOG",
            "=" * 70,
            "",
            f"Session ID: {self.session_id}",
            f"Started: {self.started_at.strftime('%Y-%m-%d %H:%M:%S')}",
            f"Ended: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "",
            "-" * 70,
            "                    CONFIGURATION",
            "-" * 70,
            "",
            f"Region: {self.region_name}",
            f"Mode: {self.download_mode}",
            f"Output Folder: {self.output_folder}",
        ]
        
        if self.aoi_file_name:
            lines.append(f"AOI File: {self.aoi_file_name}")
        
        if self.fire_period:
            lines.append(f"\nFire Period: {self.fire_period[0].strftime('%d.%m.%Y')} - {self.fire_period[1].strftime('%d.%m.%Y')}")
        
        if self.comparison_config:
            lines.extend([
                "\nComparison Settings:",
                f"  - Previous months: {self.comparison_config.include_previous_months} (count: {self.comparison_config.previous_months_count})",
                f"  - Previous year: {self.comparison_config.include_previous_year} (years: {self.comparison_config.previous_years_count})"
            ])
        
        lines.extend([
            "",
            "-" * 70,
            "                    DATA SOURCES",
            "-" * 70,
            ""
        ])
        
        for source in self.data_sources:
            lines.append(f"• {source}")
        
        lines.extend([
            "",
            "-" * 70,
            "                    DOWNLOADED FILES",
            "-" * 70,
            ""
        ])
        
        if self.firms_data_file:
            lines.append(f"FIRMS Data: {self.firms_data_file}")
        if self.boundary_file:
            lines.append(f"Boundary File: {self.boundary_file}")
        
        lines.append(f"\nTotal Files: {len(self.downloaded_files)}")
        total_size = sum(f.get('size_bytes', 0) for f in self.downloaded_files)
        lines.append(f"Total Size: {total_size / 1024 / 1024:.2f} MB")
        
        for f in self.downloaded_files:
            lines.append(f"\n  • {f['filename']}")
            lines.append(f"    Type: {f['type']} | Size: {f.get('size_bytes', 0) / 1024:.1f} KB")
        
        if self.errors:
            lines.extend([
                "",
                "-" * 70,
                "                    ERRORS",
                "-" * 70,
                ""
            ])
            for err in self.errors:
                lines.append(f"  {err}")
        
        lines.extend([
            "",
            "=" * 70,
            "                    END OF LOG",
            "=" * 70
        ])
        
        return "\n".join(lines)
    
    def save(self, folder: str):
        """Зберігає лог у файл"""
        log_path = os.path.join(folder, 'download_log.txt')
        with open(log_path, 'w', encoding='utf-8') as f:
            f.write(self.to_log_string())
        
        # Також зберігаємо JSON версію (без ключів)
        json_path = os.path.join(folder, 'download_log.json')
        log_dict = {
            'session_id': self.session_id,
            'started_at': self.started_at.isoformat(),
            'ended_at': datetime.now().isoformat(),
            'region_name': self.region_name,
            'download_mode': self.download_mode,
            'output_folder': self.output_folder,
            'downloaded_files': self.downloaded_files,
            'errors': self.errors,
            'data_sources': self.data_sources
        }
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(log_dict, f, indent=2, ensure_ascii=False)

## API Клієнти

In [ ]:
class GeoBoundariesClient:
    """Клієнт для отримання адміністративних меж з GeoBoundaries"""
    
    BOUNDARIES_URLS = [
        "https://raw.githubusercontent.com/slawomirmatuszak/ukrainian_geodata/main/regiony.geojson",
        "https://raw.githubusercontent.com/EugeneBorshch/ukraine_geojson/master/ukraine_regions.geojson",
    ]
    
    UA_NAMES = {
        'vinnytsia': 'Вінницька область', 'vinnyts\'ka': 'Вінницька область',
        'volyn': 'Волинська область', 'volyns\'ka': 'Волинська область',
        'dnipropetrovsk': 'Дніпропетровська область', 'dnipropetrovs\'ka': 'Дніпропетровська область',
        'donetsk': 'Донецька область', 'donets\'ka': 'Донецька область',
        'zhytomyr': 'Житомирська область', 'zhytomyrs\'ka': 'Житомирська область',
        'zakarpattia': 'Закарпатська область', 'zakarpats\'ka': 'Закарпатська область',
        'transcarpathia': 'Закарпатська область',
        'zaporizhzhia': 'Запорізька область', 'zaporiz\'ka': 'Запорізька область',
        'ivano-frankivsk': 'Івано-Франківська область', 'ivano-frankivs\'ka': 'Івано-Франківська область',
        'kyiv': 'Київська область', 'kyivs\'ka': 'Київська область', 'kiev': 'Київська область',
        'kirovohrad': 'Кіровоградська область', 'kirovohrads\'ka': 'Кіровоградська область',
        'luhansk': 'Луганська область', 'luhans\'ka': 'Луганська область',
        'lviv': 'Львівська область', 'l\'vivs\'ka': 'Львівська область',
        'mykolaiv': 'Миколаївська область', 'mykolaivs\'ka': 'Миколаївська область',
        'odesa': 'Одеська область', 'odes\'ka': 'Одеська область', 'odessa': 'Одеська область',
        'poltava': 'Полтавська область', 'poltavs\'ka': 'Полтавська область',
        'rivne': 'Рівненська область', 'rivnens\'ka': 'Рівненська область',
        'sumy': 'Сумська область', 'sums\'ka': 'Сумська область',
        'ternopil': 'Тернопільська область', 'ternopil\'s\'ka': 'Тернопільська область',
        'kharkiv': 'Харківська область', 'kharkivs\'ka': 'Харківська область',
        'kherson': 'Херсонська область', 'khersons\'ka': 'Херсонська область',
        'khmelnytskyi': 'Хмельницька область', 'khmel\'nyts\'ka': 'Хмельницька область',
        'cherkasy': 'Черкаська область', 'cherkas\'ka': 'Черкаська область',
        'chernivtsi': 'Чернівецька область', 'chernivets\'ka': 'Чернівецька область',
        'chernihiv': 'Чернігівська область', 'chernihivs\'ka': 'Чернігівська область',
        'crimea': 'АР Крим', 'krym': 'АР Крим',
        'sevastopol': 'м. Севастополь', 'kyiv city': 'м. Київ', 'misto kyiv': 'м. Київ',
    }
    
    @classmethod
    def get_ukraine_oblasts(cls) -> gpd.GeoDataFrame:
        try:
            api_url = "https://www.geoboundaries.org/api/current/gbOpen/UKR/ADM1/"
            response = requests.get(api_url, timeout=30)
            response.raise_for_status()
            api_data = response.json()
            geojson_url = api_data.get('gjDownloadURL') or api_data.get('simplifiedGeometryGeoJSON')
            if geojson_url:
                gdf = gpd.read_file(geojson_url)
                gdf = gdf.set_crs(epsg=4326, allow_override=True)
                print(f"✅ Завантажено {len(gdf)} областей з geoBoundaries")
                return cls._process_oblasts(gdf)
        except Exception as e:
            print(f"⚠️ geoBoundaries недоступний: {e}")
        
        for url in cls.BOUNDARIES_URLS:
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                if 'text/html' in response.headers.get('content-type', ''):
                    continue
                data = response.json()
                if 'features' in data:
                    gdf = gpd.GeoDataFrame.from_features(data['features'])
                    gdf = gdf.set_crs(epsg=4326, allow_override=True)
                    print(f"✅ Завантажено {len(gdf)} областей")
                    return cls._process_oblasts(gdf)
            except:
                continue
        raise Exception("Не вдалося завантажити межі областей!")
    
    @classmethod
    def _process_oblasts(cls, gdf):
        name_col = None
        for col in ['shapeName', 'name', 'NAME_1', 'ADM1_EN', 'region']:
            if col in gdf.columns:
                name_col = col
                break
        if not name_col:
            name_col = gdf.columns[0]
        
        def get_ua_name(en_name):
            if pd.isna(en_name):
                return 'Невідома область'
            name_lower = str(en_name).lower().strip()
            if name_lower in cls.UA_NAMES:
                return cls.UA_NAMES[name_lower]
            for key, ua_name in cls.UA_NAMES.items():
                if key in name_lower or name_lower in key:
                    return ua_name
            if any(c in en_name for c in 'абвгдеєжзиіїйклмнопрстуфхцчшщьюя'):
                return en_name
            return f"{en_name} область"
        
        gdf['name_en'] = gdf[name_col].astype(str)
        gdf['name_ua'] = gdf['name_en'].apply(get_ua_name)
        gdf['display_name'] = gdf['name_ua']
        return gdf.sort_values('name_ua').reset_index(drop=True)


class FIRMSClient:
    """
    Клієнт для NASA FIRMS API
    
    Документація: https://firms.modaps.eosdis.nasa.gov/api/area/
    
    Формат запиту:
    - NRT (останні ~60 днів): /api/area/csv/{MAP_KEY}/{SOURCE}/{AREA}/{DAY_RANGE}
    - Archive: /api/area/csv/{MAP_KEY}/{SOURCE}/{AREA}/{DAY_RANGE}/{DATE}
    
    Джерела даних:
    - MODIS_NRT / MODIS_SP - роздільна здатність ~1км
    - VIIRS_SNPP_NRT / VIIRS_SNPP_SP - роздільна здатність ~375м
    - VIIRS_NOAA20_NRT / VIIRS_NOAA20_SP
    - VIIRS_NOAA21_NRT / VIIRS_NOAA21_SP
    """
    
    AREA_API_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
    DATA_AVAILABILITY_URL = "https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv"
    
    # NRT доступний приблизно 60 днів, SP - для архівних даних
    NRT_DAYS_LIMIT = 60
    
    SOURCES_FOR_UI = [
        ('MODIS (рекомендовано для України)', 'MODIS'),
        ('VIIRS S-NPP (375м)', 'VIIRS_SNPP'),
        ('VIIRS NOAA-20 (375м)', 'VIIRS_NOAA20'),
        ('VIIRS NOAA-21 (375м)', 'VIIRS_NOAA21'),
    ]
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self._data_availability = None
    
    def check_data_availability(self) -> pd.DataFrame:
        """Перевіряє доступність даних для всіх джерел"""
        if self._data_availability is not None:
            return self._data_availability
        try:
            url = f"{self.DATA_AVAILABILITY_URL}/{self.api_key}/all"
            print(f"   🔍 Перевірка доступності даних FIRMS...")
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            from io import StringIO
            df = pd.read_csv(StringIO(response.text))
            self._data_availability = df
            print(f"   ✅ Доступні джерела:")
            for _, row in df.iterrows():
                print(f"      • {row['data_id']}: {row['min_date']} - {row['max_date']}")
            return df
        except Exception as e:
            print(f"   ⚠️ Не вдалося перевірити доступність: {e}")
            return pd.DataFrame()
    
    def _determine_source_type(self, query_date: datetime, source: str) -> Tuple[str, bool]:
        """
        Визначає тип джерела (NRT чи SP) та чи потрібен архівний запит
        
        Returns:
            Tuple[str, bool]: (повне ім'я джерела, чи це архівний запит)
        """
        today = datetime.now()
        days_ago = (today - query_date).days
        
        # Якщо дані свіжі (< 60 днів) - використовуємо NRT
        if days_ago < self.NRT_DAYS_LIMIT:
            return f"{source}_NRT", False
        else:
            # Для старіших даних - SP (Standard Processing / архів)
            return f"{source}_SP", True
    
    def get_fires(self, bbox, start_date: datetime, end_date: datetime, 
                  source: str = 'MODIS', oblast_geometry=None) -> gpd.GeoDataFrame:
        """
        Отримує дані про пожежі з FIRMS API
        
        Args:
            bbox: Bounding box (west, south, east, north)
            start_date: Початкова дата
            end_date: Кінцева дата
            source: Джерело даних (MODIS, VIIRS_SNPP, etc.)
            oblast_geometry: Геометрія області для фільтрації (опційно)
        
        Returns:
            GeoDataFrame з точками пожеж
        """
        # Форматуємо bbox як рядок
        area = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}"
        
        all_fires = []
        current_date = start_date
        
        # Максимальний діапазон для одного запиту - 10 днів
        max_days_per_request = 10
        
        print(f"   📡 Джерело: {source}")
        print(f"   📅 Період: {start_date.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')}")
        
        while current_date <= end_date:
            # Визначаємо кінець поточного чанку
            chunk_end = min(current_date + timedelta(days=max_days_per_request - 1), end_date)
            chunk_days = (chunk_end - current_date).days + 1
            
            # Визначаємо тип джерела для цієї дати
            full_source, is_archive = self._determine_source_type(current_date, source)
            
            # Формуємо URL
            if is_archive:
                # Архівний запит з датою
                url = f"{self.AREA_API_URL}/{self.api_key}/{full_source}/{area}/{chunk_days}/{current_date.strftime('%Y-%m-%d')}"
            else:
                # NRT запит без дати (останні N днів)
                url = f"{self.AREA_API_URL}/{self.api_key}/{full_source}/{area}/{chunk_days}"
            
            print(f"   📥 {current_date.strftime('%Y-%m-%d')} - {chunk_end.strftime('%Y-%m-%d')} ({full_source})...", end=" ")
            
            try:
                response = requests.get(url, timeout=60)
                
                if response.status_code == 200:
                    text = response.text.strip()
                    
                    # Перевіряємо чи це валідні CSV дані
                    if text and not text.startswith('<!') and not text.startswith('{'):
                        from io import StringIO
                        df = pd.read_csv(StringIO(text))
                        
                        if not df.empty and 'latitude' in df.columns:
                            all_fires.append(df)
                            print(f"✓ {len(df)} точок")
                        else:
                            print("- немає даних")
                    else:
                        print("- немає даних")
                elif response.status_code == 400:
                    print(f"⚠️ Невірний запит")
                elif response.status_code == 401:
                    print(f"⚠️ Невірний API ключ")
                else:
                    print(f"⚠️ HTTP {response.status_code}")
                    
            except requests.exceptions.Timeout:
                print(f"⚠️ Таймаут")
            except Exception as e:
                print(f"⚠️ {str(e)[:40]}")
            
            # Переходимо до наступного чанку
            current_date = chunk_end + timedelta(days=1)
        
        # Об'єднуємо всі результати
        if not all_fires:
            print(f"   ⚠️ Пожеж не знайдено за вказаний період")
            return gpd.GeoDataFrame()
        
        combined_df = pd.concat(all_fires, ignore_index=True)
        
        # Фільтруємо за датою
        if 'acq_date' in combined_df.columns:
            combined_df['acq_date'] = pd.to_datetime(combined_df['acq_date'])
            combined_df = combined_df[
                (combined_df['acq_date'] >= start_date) & 
                (combined_df['acq_date'] <= end_date)
            ]
        
        if combined_df.empty:
            return gpd.GeoDataFrame()
        
        # Створюємо GeoDataFrame
        gdf = gpd.GeoDataFrame(
            combined_df,
            geometry=gpd.points_from_xy(combined_df.longitude, combined_df.latitude),
            crs="EPSG:4326"
        )
        
        # Фільтруємо за геометрією області якщо передана
        if oblast_geometry is not None:
            gdf = gdf[gdf.within(oblast_geometry)]
        
        print(f"   ✅ Всього знайдено: {len(gdf)} точок пожеж")
        
        return gdf
    
    def quick_count(self, bbox, start_date: datetime, end_date: datetime, 
                    source: str = 'MODIS') -> int:
        """
        Швидкий підрахунок кількості точок пожеж без повного завантаження
        Використовується для перевірки доступності
        """
        area = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}"
        total_days = (end_date - start_date).days + 1
        
        # Для швидкої перевірки беремо весь період одним запитом (до 10 днів)
        # або розбиваємо на частини
        full_source, is_archive = self._determine_source_type(start_date, source)
        
        if is_archive:
            url = f"{self.AREA_API_URL}/{self.api_key}/{full_source}/{area}/{min(total_days, 10)}/{start_date.strftime('%Y-%m-%d')}"
        else:
            url = f"{self.AREA_API_URL}/{self.api_key}/{full_source}/{area}/{min(total_days, 10)}"
        
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                text = response.text.strip()
                if text and not text.startswith('<!') and not text.startswith('{'):
                    from io import StringIO
                    df = pd.read_csv(StringIO(text))
                    if not df.empty and 'latitude' in df.columns:
                        return len(df)
            return 0
        except:
            return -1  # Помилка
    
    @staticmethod
    def cluster_fires(fires_gdf: gpd.GeoDataFrame, buffer_km: float = 2.0) -> List[Dict]:
        """
        Кластеризує точки пожеж у полігони
        
        Args:
            fires_gdf: GeoDataFrame з точками пожеж
            buffer_km: Радіус буфера в кілометрах (як в GEE коді - 5км для study_area)
        
        Returns:
            Список кластерів з id, bbox та geometry
        """
        if fires_gdf.empty:
            return []
        
        # Проектуємо в метри для буферизації
        fires_proj = fires_gdf.to_crs(epsg=3857)
        
        # Створюємо буфер навколо кожної точки
        buffered = fires_proj.buffer(buffer_km * 1000)
        
        # Об'єднуємо перекриваючі буфери
        dissolved = buffered.unary_union
        
        # Розбиваємо на окремі полігони
        clusters = []
        if dissolved.geom_type == 'Polygon':
            geoms = [dissolved]
        elif dissolved.geom_type == 'MultiPolygon':
            geoms = list(dissolved.geoms)
        else:
            geoms = []
        
        for i, geom in enumerate(geoms):
            # Конвертуємо назад в WGS84
            cluster_gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:3857").to_crs("EPSG:4326")
            cluster_geom = cluster_gdf.iloc[0].geometry
            
            # Рахуємо кількість точок в кластері
            fires_in_cluster = fires_gdf[fires_gdf.within(cluster_geom)]
            
            clusters.append({
                'id': f'cluster_{i+1}',
                'bbox': tuple(cluster_gdf.total_bounds),
                'geometry': cluster_geom,
                'n_hotspots': len(fires_in_cluster)  # Як в GEE коді
            })
        
        return clusters


class PlanetClient:
    """
    Клієнт для Planet Labs Basemaps API
    Логіка завантаження як в робочому скрипті v2.3
    """
    
    BASE_URL = "https://api.planet.com/basemaps/v1"
    
    MONTHS_EN = {
        '01': 'january', '02': 'february', '03': 'march', '04': 'april',
        '05': 'may', '06': 'june', '07': 'july', '08': 'august',
        '09': 'september', '10': 'october', '11': 'november', '12': 'december'
    }
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.auth = requests.auth.HTTPBasicAuth(api_key, "")
        self._mosaics_cache = None
    
    def test_connection(self) -> bool:
        try:
            r = requests.get(f"{self.BASE_URL}/mosaics", auth=self.auth, params={"_limit": 1}, timeout=30)
            r.raise_for_status()
            print("   ✅ Planet API підключено")
            return True
        except Exception as e:
            print(f"   ❌ Planet API помилка: {e}")
            return False
    
    def get_mosaic_list(self) -> Dict[str, str]:
        if self._mosaics_cache:
            return self._mosaics_cache
        
        print("   🔍 Пошук мозаїк...")
        all_mosaics = {}
        
        for pattern in ["global_monthly", "planet_medres", "2025", "2024", "2023", "2022", "2021", "2020"]:
            try:
                r = requests.get(f"{self.BASE_URL}/mosaics", auth=self.auth,
                               params={"name__contains": pattern, "_limit": 300}, timeout=60)
                r.raise_for_status()
                for m in r.json().get("mosaics", []):
                    if m["name"] not in all_mosaics:
                        all_mosaics[m["name"]] = m["id"]
            except:
                continue
        
        self._mosaics_cache = all_mosaics
        print(f"   ✅ Знайдено {len(all_mosaics)} мозаїк")
        return all_mosaics
    
    def find_mosaic(self, year: str, month: str) -> Optional[Dict]:
        mosaics = self.get_mosaic_list()
        month_name = self.MONTHS_EN.get(month, '')
        
        patterns = [
            f"planet_medres_visual_global_monthly_{year}_{month_name}",
            f"global_monthly_{year}_{month_name}",
            f"global_monthly_{year}_{month}",
            f"{year}_{month_name}", f"{year}_{month}"
        ]
        
        for pattern in patterns:
            for name, mosaic_id in mosaics.items():
                if pattern.lower() in name.lower():
                    return {'name': name, 'id': mosaic_id}
        return None
    
    def get_quads_for_aoi(self, mosaic_id: str, aoi_gdf: gpd.GeoDataFrame) -> List[Dict]:
        """Отримує квадрати як в робочому скрипті - повертає оригінальні об'єкти API"""
        bounds = aoi_gdf.total_bounds
        bbox = ",".join(str(v) for v in bounds)
        
        params = {"bbox": bbox, "_page_size": 500}
        response = requests.get(f"{self.BASE_URL}/mosaics/{mosaic_id}/quads",
                              auth=self.auth, params=params, timeout=60)
        response.raise_for_status()
        
        quads_response = response.json()
        items = quads_response.get("items", [])
        
        # Пагінація
        page = 1
        while quads_response["_links"].get("_next"):
            page += 1
            if page % 5 == 0:
                print(f"      📄 Сторінка {page}...")
            response = requests.get(quads_response["_links"]["_next"], auth=self.auth, timeout=60)
            response.raise_for_status()
            quads_response = response.json()
            items.extend(quads_response.get("items", []))
        
        print(f"      📊 API повернув {len(items)} квадратів")
        
        if not items:
            return []
        
        # Фільтруємо за AOI
        quads_gdf = gpd.GeoDataFrame([
            {"quad_id": q["id"], "geometry": box(*q["bbox"])}
            for q in items
        ]).set_crs(epsg=4326)
        
        filtered = gpd.sjoin(quads_gdf, aoi_gdf[['geometry']], how="inner", predicate="intersects")
        filtered_ids = set(filtered['quad_id'].values)
        
        # Повертаємо оригінальні об'єкти з URL
        result = [q for q in items if q["id"] in filtered_ids]
        print(f"      ✅ Після фільтрації: {len(result)} квадратів")
        
        return result
    
    def download_tiles(self, quads: List[Dict], output_dir: str) -> Tuple[int, int, int]:
        """Завантаження плиток з правильною аутентифікацією (як в робочому скрипті)"""
        
        # Перевірка існуючих
        existing = {f.replace('.tif', '') for f in os.listdir(output_dir) if f.endswith('.tif')}
        to_download = [q for q in quads if q["id"] not in existing]
        
        if existing:
            print(f"      ✅ Вже завантажено: {len(existing)}")
        
        if not to_download:
            print(f"      🎉 Всі файли є!")
            return len(existing), 0, 0
        
        print(f"      📥 Потрібно завантажити: {len(to_download)}")
        
        # Сесія з headers (як в робочому скрипті)
        session = requests.Session()
        session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        })
        
        downloaded = 0
        failed = 0
        
        for i, quad in enumerate(to_download, 1):
            quad_id = quad["id"]
            output_file = os.path.join(output_dir, f"{quad_id}.tif")
            
            if os.path.exists(output_file):
                downloaded += 1
                continue
            
            download_url = quad["_links"]["download"]
            
            # КЛЮЧОВА РІЗНИЦЯ: додаємо api_key до URL
            if "api_key=" not in download_url:
                separator = "&" if "?" in download_url else "?"
                download_url = f"{download_url}{separator}api_key={self.api_key}"
            
            success = False
            last_error = ""
            
            for attempt in range(3):
                try:
                    response = session.get(download_url, allow_redirects=True, timeout=180)
                    response.raise_for_status()
                    
                    content = response.content
                    
                    # Перевірка розміру
                    if len(content) < 1000:
                        raise Exception(f"Малий розмір: {len(content)}")
                    
                    with open(output_file, 'wb') as f:
                        f.write(content)
                    
                    downloaded += 1
                    success = True
                    
                    if i % 5 == 0 or i == len(to_download):
                        print(f"      📥 [{i}/{len(to_download)}] {quad_id}: {len(content)/1024/1024:.1f} MB")
                    
                    break
                    
                except Exception as e:
                    last_error = str(e)
                    if attempt < 2:
                        time.sleep(3 + attempt * 3)
            
            if not success:
                failed += 1
                print(f"      ❌ [{i}/{len(to_download)}] {quad_id}: {last_error[:50]}")
            
            time.sleep(0.3)
        
        session.close()
        return len(existing), downloaded, failed
    
    def download_area(self, aoi_gdf: gpd.GeoDataFrame, year: str, month: str, 
                      output_dir: str) -> Dict:
        """Завантажує всі плитки для області"""
        stats = {'total': 0, 'exists': 0, 'downloaded': 0, 'failed': 0}
        
        mosaic = self.find_mosaic(year, month)
        if not mosaic:
            print(f"   ❌ Мозаїка для {year}-{month} не знайдена")
            return stats
        
        print(f"   📅 {year}-{month}: {mosaic['name']}")
        
        # Папка для періоду
        period_dir = os.path.join(output_dir, f"{year}_{month}")
        os.makedirs(period_dir, exist_ok=True)
        
        print(f"   🔍 Пошук квадратів...")
        
        # Отримуємо свіжі квадрати з URL
        quads = self.get_quads_for_aoi(mosaic['id'], aoi_gdf)
        
        if not quads:
            print(f"   ⚠️ Квадратів не знайдено")
            return stats
        
        stats['total'] = len(quads)
        
        # Індекс
        index_gdf = gpd.GeoDataFrame([
            {"quad_id": q["id"], "geometry": box(*q["bbox"])}
            for q in quads
        ]).set_crs(epsg=4326)
        index_gdf.to_file(os.path.join(period_dir, "index.geojson"), driver='GeoJSON')
        
        # Завантаження
        exists, downloaded, failed = self.download_tiles(quads, period_dir)
        
        stats['exists'] = exists
        stats['downloaded'] = downloaded
        stats['failed'] = failed
        
        print(f"   ✅ Результат: існувало {exists}, завантажено {downloaded}, помилок {failed}")
        return stats


import time

## Допоміжні функції для роботи з файлами

In [ ]:
def parse_uploaded_file(file_path: str) -> gpd.GeoDataFrame:
    """Парсить завантажений файл (GeoJSON, Shapefile, KML, KMZ)"""
    ext = os.path.splitext(file_path)[1].lower()
    
    if ext in ['.geojson', '.json']:
        return gpd.read_file(file_path)
    elif ext == '.shp':
        return gpd.read_file(file_path)
    elif ext == '.kml':
        gpd.io.file.fiona.drvsupport.supported_drivers['KML'] = 'rw'
        return gpd.read_file(file_path, driver='KML')
    elif ext == '.kmz':
        # KMZ - це ZIP з KML всередині
        with zipfile.ZipFile(file_path, 'r') as z:
            for name in z.namelist():
                if name.endswith('.kml'):
                    kml_content = z.read(name)
                    gpd.io.file.fiona.drvsupport.supported_drivers['KML'] = 'rw'
                    return gpd.read_file(BytesIO(kml_content), driver='KML')
        raise ValueError("KML файл не знайдено в KMZ архіві")
    else:
        raise ValueError(f"Непідтримуваний формат файлу: {ext}")


def create_folder_structure(base_path: str, region_name: str, mode: str,
                           fire_period: Optional[Tuple[datetime, datetime]] = None,
                           comparison_config: Optional[ComparisonConfig] = None) -> Dict[str, str]:
    """Створює структуру папок для завантаження"""
    
    # Санітизуємо назву
    safe_name = region_name.replace(' ', '_').replace('/', '_').replace('\\', '_')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    if fire_period:
        folder_name = f"{safe_name}_{mode}_{fire_period[0].strftime('%Y%m%d')}_to_{fire_period[1].strftime('%Y%m%d')}_{timestamp}"
    else:
        folder_name = f"{safe_name}_{mode}_{timestamp}"
    
    session_folder = os.path.join(base_path, folder_name)
    os.makedirs(session_folder, exist_ok=True)
    
    folders = {
        'session': session_folder,
        'data': os.path.join(session_folder, 'data'),
        'logs': os.path.join(session_folder, 'logs')
    }
    
    # Створюємо підпапки
    for folder in folders.values():
        os.makedirs(folder, exist_ok=True)
    
    # Для режиму пожеж - додаткові папки
    if mode == 'fire_areas' and fire_period and comparison_config:
        folders['fire_period'] = os.path.join(
            session_folder,
            f"fire_period_{fire_period[0].strftime('%Y%m%d')}_to_{fire_period[1].strftime('%Y%m%d')}"
        )
        os.makedirs(folders['fire_period'], exist_ok=True)
        
        # Папки для порівняльних періодів
        for period in comparison_config.generate_periods(fire_period[0], fire_period[1]):
            period_folder = os.path.join(
                session_folder,
                f"comparison_{period['folder_suffix']}_{period['start'].strftime('%Y%m%d')}_to_{period['end'].strftime('%Y%m%d')}"
            )
            os.makedirs(period_folder, exist_ok=True)
            folders[f"comparison_{period['folder_suffix']}"] = period_folder
    
    return folders


def save_geodataframe(gdf: gpd.GeoDataFrame, folder: str, filename: str) -> str:
    """Зберігає GeoDataFrame у файл"""
    filepath = os.path.join(folder, filename)
    gdf.to_file(filepath, driver='GeoJSON')
    return filepath

## Головний інтерфейс

In [ ]:
class ForestFireUI:
    """Головний клас інтерфейсу користувача"""
    
    def __init__(self):
        self.oblasts_gdf = None
        self.selected_oblast = None
        self.aoi_gdf = None
        self.log = None
        
        self._create_widgets()
        self._load_oblasts()
    
    def _create_widgets(self):
        """Створює всі віджети"""
        
        style = {'description_width': '150px'}
        layout = widgets.Layout(width='500px')
        
        self.oblast_dropdown = widgets.Dropdown(
            options=[('Завантаження...', None)],
            description='Область:',
            style=style,
            layout=layout
        )
        
        self.mode_radio = widgets.RadioButtons(
            options=[
                ('Вся область - Завантажити супутникові знімки', 'full_oblast'),
                ('Зона інтересу - Завантажити власну ділянку', 'aoi'),
                ('Лише пожежі - Завантажити тільки території з пожежами', 'fire_areas')
            ],
            value='fire_areas',
            description='Режим:',
            style=style,
            layout=widgets.Layout(width='600px')
        )
        self.mode_radio.observe(self._on_mode_change, names='value')
        
        today = datetime.now()
        month_ago = today - timedelta(days=30)
        
        self.fire_start_date = widgets.DatePicker(
            description='Початок:',
            value=month_ago.date(),
            style=style
        )
        
        self.fire_end_date = widgets.DatePicker(
            description='Кінець:',
            value=today.date(),
            style=style
        )
        
        self.firms_source = widgets.Dropdown(
            options=FIRMSClient.SOURCES_FOR_UI,
            value='MODIS',
            description='Джерело FIRMS:',
            style=style,
            layout=layout
        )
        
        self.include_prev_months = widgets.Checkbox(value=True, description='Включити попередні місяці', style=style)
        self.prev_months_count = widgets.IntSlider(value=2, min=1, max=6, description='Кількість місяців:', style=style)
        self.include_prev_year = widgets.Checkbox(value=True, description='Ті ж місяці в попередньому році', style=style)
        self.prev_years_count = widgets.IntSlider(value=1, min=1, max=3, description='Кількість років:', style=style)
        
        self.fire_options_box = widgets.VBox([
            widgets.HTML('<h4>🔥 Налаштування виявлення пожеж:</h4>'),
            widgets.HBox([self.fire_start_date, self.fire_end_date]),
            self.firms_source,
            widgets.HTML('<br><b>Порівняльні знімки:</b>'),
            self.include_prev_months, self.prev_months_count,
            self.include_prev_year, self.prev_years_count
        ])
        
        self.file_upload = widgets.FileUpload(accept='.geojson,.json,.shp,.kml,.kmz,.zip', multiple=False, description='Завантажити файл')
        self.file_upload.observe(self._on_file_upload, names='value')
        self.aoi_status = widgets.HTML('<i>Файл не завантажено</i>')
        
        self.aoi_options_box = widgets.VBox([
            widgets.HTML('<h4>📍 Завантажте файл з межами:</h4>'),
            self.file_upload, self.aoi_status,
            widgets.HTML('<small>Підтримувані формати: GeoJSON, Shapefile, KML, KMZ</small>')
        ])
        self.aoi_options_box.layout.display = 'none'
        
        current_year = datetime.now().year
        self.years_select = widgets.SelectMultiple(
            options=[str(y) for y in range(current_year, 2018, -1)],
            value=[str(current_year)], description='Роки:', style=style, rows=6
        )
        
        months = [('01 - Січень', '01'), ('02 - Лютий', '02'), ('03 - Березень', '03'),
                  ('04 - Квітень', '04'), ('05 - Травень', '05'), ('06 - Червень', '06'),
                  ('07 - Липень', '07'), ('08 - Серпень', '08'), ('09 - Вересень', '09'),
                  ('10 - Жовтень', '10'), ('11 - Листопад', '11'), ('12 - Грудень', '12')]
        self.months_select = widgets.SelectMultiple(
            options=months, value=[f"{datetime.now().month:02d}"],
            description='Місяці:', style=style, rows=12
        )
        
        self.full_oblast_options_box = widgets.VBox([
            widgets.HTML('<h4>🛰️ Planet: Оберіть періоди для завантаження:</h4>'),
            widgets.HTML('<small>Ctrl+Click для вибору кількох</small>'),
            widgets.HBox([self.years_select, self.months_select])
        ])
        self.full_oblast_options_box.layout.display = 'none'
        
        self.planet_key = widgets.Password(description='Planet API Key:', placeholder='Обов\'язково для знімків', style=style, layout=layout)
        self.firms_key = widgets.Password(description='FIRMS MAP Key:', placeholder='Для режиму "Пожежі"', style=style, layout=layout)
        self.output_folder = widgets.Text(
            description='Папка виводу:',
            value='/content/drive/MyDrive/ForestFireData' if IN_COLAB else './output',
            style=style, layout=layout
        )
        
        self.check_btn = widgets.Button(description='🔍 Перевірити', button_style='info', layout=widgets.Layout(width='180px', height='40px'))
        self.check_btn.on_click(self._on_check_click)
        
        self.start_button = widgets.Button(description='📥 Завантажити', button_style='success', layout=widgets.Layout(width='180px', height='40px'))
        self.start_button.on_click(self._on_start_click)
        
        self.progress = widgets.IntProgress(value=0, min=0, max=100, description='Прогрес:', style=style, layout=widgets.Layout(width='400px'))
        self.progress.layout.display = 'none'
        self.progress_label = widgets.HTML('')
        self.status_output = widgets.Output()
    
    def _load_oblasts(self):
        try:
            self.oblasts_gdf = GeoBoundariesClient.get_ukraine_oblasts()
            options = [(row['display_name'], idx) for idx, row in self.oblasts_gdf.iterrows()]
            self.oblast_dropdown.options = [('-- Оберіть область --', None)] + options
        except Exception as e:
            self.oblast_dropdown.options = [(f'Помилка: {str(e)[:50]}', None)]
    
    def _on_mode_change(self, change):
        mode = change['new']
        self.fire_options_box.layout.display = 'block' if mode == 'fire_areas' else 'none'
        self.aoi_options_box.layout.display = 'block' if mode == 'aoi' else 'none'
        self.full_oblast_options_box.layout.display = 'block' if mode == 'full_oblast' else 'none'
    
    def _on_file_upload(self, change):
        if change['new']:
            try:
                uploaded = list(change['new'].values())[0]
                filename = list(change['new'].keys())[0]
                temp_path = f'/tmp/{filename}'
                with open(temp_path, 'wb') as f:
                    f.write(uploaded['content'])
                self.aoi_gdf = parse_uploaded_file(temp_path)
                bounds = self.aoi_gdf.total_bounds
                self.aoi_status.value = f'<span style="color: green;">✓ {filename}</span><br><small>Bbox: ({bounds[0]:.4f}, {bounds[1]:.4f}) - ({bounds[2]:.4f}, {bounds[3]:.4f})</small>'
            except Exception as e:
                self.aoi_status.value = f'<span style="color: red;">Помилка: {str(e)}</span>'
                self.aoi_gdf = None
    
    def _update_progress(self, value, label=""):
        self.progress.value = value
        if label:
            self.progress_label.value = f'<small>{label}</small>'
    
    def _on_check_click(self, button):
        """Перевірка доступності даних з підрахунком пожеж"""
        with self.status_output:
            clear_output()
            
            print("=" * 55)
            print("🔍 ПЕРЕВІРКА ДОСТУПНОСТІ ДАНИХ")
            print("=" * 55)
            
            # Отримуємо вибрану область
            oblast = None
            oblast_name = "не вибрано"
            if self.oblast_dropdown.value is not None and self.oblasts_gdf is not None:
                oblast = self.oblasts_gdf.loc[self.oblast_dropdown.value]
                oblast_name = oblast['display_name']
            
            print(f"\n📍 Область: {oblast_name}")
            
            # Перевірка Planet
            print("\n" + "-" * 55)
            print("🛰️ PLANET API")
            print("-" * 55)
            if self.planet_key.value:
                try:
                    planet = PlanetClient(self.planet_key.value)
                    if planet.test_connection():
                        mosaics = planet.get_mosaic_list()
                        print(f"   ✅ Доступно {len(mosaics)} мозаїк")
                        
                        # Показуємо останні доступні
                        recent = sorted([m for m in mosaics.keys() if '2024' in m or '2025' in m])[-5:]
                        print(f"   📅 Останні: {', '.join(recent[:3])}...")
                except Exception as e:
                    print(f"   ❌ Помилка: {e}")
            else:
                print("   ⚠️ API ключ не вказано")
            
            # Перевірка FIRMS з підрахунком пожеж
            print("\n" + "-" * 55)
            print("🔥 FIRMS API (NASA)")
            print("-" * 55)
            if self.firms_key.value:
                try:
                    firms = FIRMSClient(self.firms_key.value)
                    firms.check_data_availability()
                    
                    # Підрахунок пожеж для обраної області
                    if oblast is not None:
                        start_date = datetime.combine(self.fire_start_date.value, datetime.min.time())
                        end_date = datetime.combine(self.fire_end_date.value, datetime.max.time())
                        source = self.firms_source.value
                        
                        print(f"\n   📊 Підрахунок пожеж для {oblast_name}:")
                        print(f"      Період: {start_date.strftime('%d.%m.%Y')} - {end_date.strftime('%d.%m.%Y')}")
                        print(f"      Джерело: {source}")
                        
                        bbox = oblast.geometry.bounds
                        fires_gdf = firms.get_fires(bbox, start_date, end_date, source=source)
                        
                        if not fires_gdf.empty:
                            # Фільтруємо в межах області
                            fires_in_oblast = fires_gdf[fires_gdf.within(oblast.geometry)]
                            
                            print(f"\n   🔥 РЕЗУЛЬТАТ: {len(fires_in_oblast)} точок горіння")
                            print(f"      (у bbox знайдено {len(fires_gdf)}, в межах області {len(fires_in_oblast)})")
                            
                            if not fires_in_oblast.empty and 'acq_date' in fires_in_oblast.columns:
                                dates = fires_in_oblast['acq_date'].dt.strftime('%Y-%m-%d').value_counts().head(5)
                                print(f"      Топ дати: {dict(dates)}")
                        else:
                            print(f"\n   ⚠️ Пожеж не знайдено за вказаний період")
                    else:
                        print("\n   ℹ️ Оберіть область для підрахунку пожеж")
                        
                except Exception as e:
                    print(f"   ❌ Помилка: {e}")
            else:
                print("   ⚠️ API ключ не вказано")
            
            print("\n" + "=" * 55)
            print("✅ Перевірка завершена")
            print("=" * 55)
    
    def _on_start_click(self, button):
        with self.status_output:
            clear_output()
            errors = []
            
            if self.oblast_dropdown.value is None:
                errors.append('Оберіть область')
            
            mode = self.mode_radio.value
            if mode == 'aoi' and self.aoi_gdf is None:
                errors.append('Завантажте файл з межами')
            if mode == 'fire_areas':
                if not self.firms_key.value:
                    errors.append('Введіть FIRMS API ключ')
                if not self.planet_key.value:
                    errors.append('Введіть Planet API ключ')
            if mode == 'full_oblast':
                if not self.planet_key.value:
                    errors.append('Введіть Planet API ключ')
                if not self.years_select.value:
                    errors.append('Оберіть хоча б один рік')
                if not self.months_select.value:
                    errors.append('Оберіть хоча б один місяць')
            
            if errors:
                for err in errors:
                    print(f'⚠️ {err}')
                return
            
            self._run_download()
    
    def _run_download(self):
        mode = self.mode_radio.value
        oblast = self.oblasts_gdf.loc[self.oblast_dropdown.value]
        
        self.progress.value = 0
        self.progress.layout.display = 'block'
        self.progress_label.value = ''
        
        self.log = SessionLog(region_name=oblast['display_name'], download_mode=mode)
        self.log.data_sources.append('GeoBoundaries (https://www.geoboundaries.org)')
        
        try:
            if mode == 'fire_areas':
                self._download_fire_areas(oblast)
            elif mode == 'full_oblast':
                self._download_full_oblast(oblast)
            elif mode == 'aoi':
                self._download_aoi(oblast)
            
            print('\n' + '=' * 50)
            print('✅ ЗАВАНТАЖЕННЯ ЗАВЕРШЕНО!')
            print('=' * 50)
            print(f'📁 Файли: {self.log.output_folder}')
            
        except Exception as e:
            self.log.add_error(str(e))
            print(f'\n❌ Помилка: {str(e)}')
            import traceback
            traceback.print_exc()
        finally:
            if self.log.output_folder:
                logs_folder = os.path.join(self.log.output_folder, 'logs')
                os.makedirs(logs_folder, exist_ok=True)
                self.log.save(logs_folder)
                print(f'📝 Лог: {logs_folder}')
            self._update_progress(100, 'Завершено')
    
    def _download_fire_areas(self, oblast):
        """Завантаження Planet знімків для територій з пожежами"""
        print('=' * 50)
        print('🔥 РЕЖИМ: ТЕРИТОРІЇ З ПОЖЕЖАМИ')
        print('=' * 50)

        start_date = datetime.combine(self.fire_start_date.value, datetime.min.time())
        end_date = datetime.combine(self.fire_end_date.value, datetime.max.time())
        firms_source = self.firms_source.value

        print(f'\n📍 Область: {oblast["display_name"]}')
        print(f'📅 Період: {start_date.strftime("%d.%m.%Y")} - {end_date.strftime("%d.%m.%Y")}')
        print(f'📡 Джерело: {firms_source}')

        comparison_config = ComparisonConfig(
            include_previous_months=self.include_prev_months.value,
            previous_months_count=self.prev_months_count.value,
            include_previous_year=self.include_prev_year.value,
            previous_years_count=self.prev_years_count.value
        )

        self.log.fire_period = (start_date, end_date)
        self.log.comparison_config = comparison_config
        self.log.data_sources.append(f'NASA FIRMS - {firms_source}')

        folders = create_folder_structure(
            self.output_folder.value, oblast.get('name_en', oblast['display_name']),
            'fire_areas', (start_date, end_date), comparison_config
        )
        self.log.output_folder = folders['session']
        print(f'\n📁 Папка: {folders["session"]}')
        self._update_progress(5, 'Створення папок...')

        oblast_gdf = gpd.GeoDataFrame([oblast], crs="EPSG:4326")
        boundary_path = save_geodataframe(oblast_gdf, folders['data'], 'oblast_boundary.geojson')
        self.log.boundary_file = boundary_path

        # КРОК 1: Пошук пожеж
        print(f'\n{"="*50}')
        print(f'📡 КРОК 1: Пошук пожеж через FIRMS')
        print(f'{"="*50}')
        self._update_progress(10, 'Пошук пожеж...')

        bbox = oblast.geometry.bounds
        firms_client = FIRMSClient(self.firms_key.value)
        firms_client.check_data_availability()
        fires_gdf = firms_client.get_fires(bbox, start_date, end_date, source=firms_source)
        self._update_progress(20, 'Обробка...')

        if fires_gdf.empty:
            print('\n⚠️ Пожеж не знайдено')
            return

        fires_in_oblast = fires_gdf[fires_gdf.within(oblast.geometry)]
        print(f'\n🔥 Знайдено {len(fires_in_oblast)} точок в області')

        if fires_in_oblast.empty:
            print('\n⚠️ Пожежі за межами області')
            return

        firms_path = save_geodataframe(fires_in_oblast, folders['data'], 'firms_fire_data.geojson')
        self.log.firms_data_file = firms_path
        self.log.add_file('firms_fire_data.geojson', 'fire_data', os.path.getsize(firms_path))

        # КРОК 2: Кластеризація
        print(f'\n{"="*50}')
        print(f'📊 КРОК 2: Кластеризація пожеж')
        print(f'{"="*50}')
        self._update_progress(25, 'Кластеризація...')

        clusters = FIRMSClient.cluster_fires(fires_in_oblast, buffer_km=2.0)
        print(f'   Кластерів: {len(clusters)}')

        if not clusters:
            return

        fire_areas_gdf = gpd.GeoDataFrame(
            {'cluster_id': [c['id'] for c in clusters], 'n_hotspots': [c['n_hotspots'] for c in clusters]},
            geometry=[c['geometry'] for c in clusters], crs="EPSG:4326"
        )
        clusters_path = save_geodataframe(fire_areas_gdf, folders['data'], 'fire_clusters.geojson')
        self.log.add_file('fire_clusters.geojson', 'fire_clusters', os.path.getsize(clusters_path))

        # КРОК 3: Planet
        print(f'\n{"="*50}')
        print(f'🛰️ КРОК 3: Завантаження Planet')
        print(f'{"="*50}')
        self._update_progress(30, 'Planet API...')
        self.log.data_sources.append('Planet Labs Basemaps')

        planet = PlanetClient(self.planet_key.value)
        if not planet.test_connection():
            print('❌ Planet API недоступний')
            return

        # Визначаємо періоди
        fire_months = set()
        current = start_date
        while current <= end_date:
            fire_months.add((str(current.year), f"{current.month:02d}"))
            current += timedelta(days=28)
        fire_months.add((str(end_date.year), f"{end_date.month:02d}"))
        fire_months = sorted(list(fire_months))

        comparison_periods = comparison_config.generate_periods(start_date, end_date)
        all_periods = []

        for year, month in fire_months:
            all_periods.append({'year': year, 'month': month, 'type': 'fire_period',
                               'description': f'Пожежі {year}-{month}', 'folder': folders.get('fire_period', folders['session'])})

        for cp in comparison_periods:
            cp_months = set()
            cur = cp['start']
            while cur <= cp['end']:
                cp_months.add((str(cur.year), f"{cur.month:02d}"))
                cur += timedelta(days=28)
            cp_months.add((str(cp['end'].year), f"{cp['end'].month:02d}"))
            folder_key = f"comparison_{cp['folder_suffix']}"
            cp_folder = folders.get(folder_key, folders['session'])
            for year, month in sorted(cp_months):
                all_periods.append({'year': year, 'month': month, 'type': cp['type'],
                                   'description': f"{cp['description']} ({year}-{month})", 'folder': cp_folder})

        print(f'\n📊 Періодів: {len(all_periods)}')
        self._update_progress(35, 'Завантаження...')

        total_stats = {'total': 0, 'exists': 0, 'downloaded': 0, 'failed': 0}
        progress_per = 60 / max(len(all_periods), 1)

        for i, period in enumerate(all_periods):
            year, month = period['year'], period['month']
            print(f'\n📥 [{i+1}/{len(all_periods)}] {period["description"]}')
            self._update_progress(35 + int(i * progress_per), f'{year}-{month}...')

            mosaic = planet.find_mosaic(year, month)
            if not mosaic:
                print(f'   ❌ Мозаїка не знайдена')
                continue

            period_dir = os.path.join(period['folder'], f'{year}_{month}')
            os.makedirs(period_dir, exist_ok=True)

            quads = planet.get_quads_for_aoi(mosaic['id'], fire_areas_gdf)
            if not quads:
                continue

            total_stats['total'] += len(quads)
            
            index_gdf = gpd.GeoDataFrame([{"quad_id": q["id"], "geometry": box(*q["bbox"])} for q in quads]).set_crs(epsg=4326)
            index_gdf.to_file(os.path.join(period_dir, "quads_index.geojson"), driver='GeoJSON')

            exists, downloaded, failed = planet.download_tiles(quads, period_dir)
            total_stats['exists'] += exists
            total_stats['downloaded'] += downloaded
            total_stats['failed'] += failed
            print(f'   ✅ {exists} існ., {downloaded} зав., {failed} пом.')

        print(f'\n{"="*50}')
        print(f'📊 ПІДСУМОК: {len(fires_in_oblast)} пожеж, {len(clusters)} кластерів')
        print(f'   Тайлів: {total_stats["total"]}, завантажено: {total_stats["downloaded"]}')
        self._update_progress(95, 'Фіналізація...')
    
    def _download_full_oblast(self, oblast):
        """Завантаження для всієї області"""
        print('=' * 50)
        print('🛰️ РЕЖИМ: ВСЯ ОБЛАСТЬ')
        print('=' * 50)
        
        years, months = list(self.years_select.value), list(self.months_select.value)
        print(f'\n📍 {oblast["display_name"]}')
        print(f'📅 {years} x {months}')
        
        oblast_name = oblast.get('name_en', oblast['display_name']).replace(' ', '_')
        folders = create_folder_structure(self.output_folder.value, oblast_name, 'full_oblast')
        self.log.output_folder = folders['session']
        self.log.data_sources.append('Planet Labs Basemaps')
        
        oblast_gdf = gpd.GeoDataFrame([oblast], crs="EPSG:4326")
        save_geodataframe(oblast_gdf, folders['data'], 'oblast_boundary.geojson')
        self._update_progress(10, 'Planet API...')
        
        planet = PlanetClient(self.planet_key.value)
        if not planet.test_connection():
            raise Exception("Planet API недоступний")
        
        available = [(y, m, planet.find_mosaic(y, m)) for y in years for m in months if planet.find_mosaic(y, m)]
        if not available:
            raise Exception("Немає мозаїк")
        
        self._update_progress(20, 'Завантаження...')
        total_stats = {'total': 0, 'downloaded': 0, 'exists': 0, 'failed': 0}
        
        for i, (year, month, mosaic) in enumerate(available):
            print(f'\n📥 [{i+1}/{len(available)}] {year}-{month}')
            self._update_progress(20 + int(70 * i / len(available)), f'{year}-{month}...')
            stats = planet.download_area(oblast_gdf, year, month, folders['session'])
            for k in total_stats:
                total_stats[k] += stats.get(k, 0)
        
        print(f'\n📊 Всього: {total_stats["total"]}, завантажено: {total_stats["downloaded"]}')
        self._update_progress(95, 'Фіналізація...')
    
    def _download_aoi(self, oblast):
        """Завантаження для AOI"""
        print('=' * 50)
        print('📍 РЕЖИМ: ЗОНА ІНТЕРЕСУ')
        print('=' * 50)
        
        self.log.aoi_file_name = 'uploaded_aoi'
        folders = create_folder_structure(self.output_folder.value, 'AOI', 'aoi')
        self.log.output_folder = folders['session']
        
        save_geodataframe(self.aoi_gdf, folders['data'], 'aoi_boundary.geojson')
        self._update_progress(20, 'Planet API...')
        
        if self.planet_key.value:
            years = list(self.years_select.value) if self.years_select.value else [str(datetime.now().year)]
            months = list(self.months_select.value) if self.months_select.value else [f"{datetime.now().month:02d}"]
            planet = PlanetClient(self.planet_key.value)
            if planet.test_connection():
                for i, (y, m) in enumerate([(y, m) for y in years for m in months]):
                    self._update_progress(20 + int(70 * i / max(len(years)*len(months), 1)), f'{y}-{m}...')
                    planet.download_area(self.aoi_gdf, y, m, folders['session'])
        self._update_progress(95, 'Фіналізація...')
    
    def display(self):
        ui = widgets.VBox([
            widgets.HTML('<h2>🔥 Forest Fire Risk RS</h2>'),
            widgets.HTML('<p>Система аналізу лісових пожеж в Україні</p><hr>'),
            widgets.HTML('<h3>1. Область:</h3>'), self.oblast_dropdown,
            widgets.HTML('<h3>2. Режим:</h3>'), self.mode_radio,
            self.fire_options_box, self.aoi_options_box, self.full_oblast_options_box,
            widgets.HTML('<h3>3. API ключі:</h3>'),
            self.planet_key, widgets.HTML('<small><a href="https://www.planet.com/account/" target="_blank">Planet API</a></small>'),
            self.firms_key, widgets.HTML('<small><a href="https://firms.modaps.eosdis.nasa.gov/api/map_key/" target="_blank">FIRMS API</a></small>'),
            widgets.HTML('<h3>4. Папка:</h3>'), self.output_folder,
            widgets.HTML('<hr>'), widgets.HBox([self.check_btn, self.start_button]),
            widgets.VBox([self.progress, self.progress_label]), self.status_output
        ])
        display(ui)
        self._on_mode_change({'new': self.mode_radio.value})

## Запуск інтерфейсу

In [ ]:
# Створюємо та відображаємо інтерфейс
ui = ForestFireUI()
ui.display()